In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml import LinearRegression
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold

In [2]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [3]:
from ipynb.fs.full.Normalization_to_import import *

Execution time: 0.8079581260681152 seconds


In [4]:
torch.cuda.empty_cache()
gc.collect()

20

In [5]:
from ipynb.fs.full.Data_Generation_to_import import *

Execution time: 0.05915093421936035 seconds


In [6]:
torch.cuda.empty_cache()
gc.collect()

0

In [7]:
class machine_learing(object):
    def __init__(self, dataset, generated_data_global_1):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['HeartDiseaseorAttack'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['HeartDiseaseorAttack'].copy())
        self.generated_data_global_1 = generated_data_global_1.copy().reset_index(drop = True)

    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def MiniBatchSGDClassifier(self):
        global mbc_global
        global mbc_predict_test_global
        global mbc_predict_global_1
        global mean_squared_error_global_mbc
        global mean_squared_log_error_global_mbc
        global median_absolute_error_global_mbc
        global r2_score_global_mbc
        global accuracy_score_global_mbc
        global kl_divergence_global_mbc
        global log_loss_global_mbc
        global roc_auc_score_global_mbc
        global nan_euclidean_distances_global_mbc
        global pairwise_distances_global_mbc
        global sparse_pairwise_distances_global_mbc 

        model = cuml.MBSGDClassifier(loss='hinge', penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, epochs=1000, tol=0.001, 
                                     shuffle=True, learning_rate='constant', eta0=0.001, power_t=0.5, batch_size=32, n_iter_no_change=5, 
                                     handle=None, verbose=False, output_type=None)
        mbc = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        mbc_global = mbc

        preds_test = model.predict(X_test_global.copy())
        mbc_predict_test_global = preds_test
        
        mean_squared_error_global_mbc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_mbc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_mbc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_mbc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_mbc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_mbc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_mbc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_mbc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_mbc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_mbc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_mbc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        mbc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
    def main(self):
        st = time.time()
        self.train_test_split()
        self.MiniBatchSGDClassifier()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [8]:
class Metrics(object):
    
    def MiniBatchSGDClassifier(self):
        print('Mini Batch SGD Classifier Coef: ')
        print(mbc_global.coef_)
        print('\n')
        print('Mini Batch SGD Classifier Intercept: ')
        print(mbc_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_mbc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_mbc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_mbc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_mbc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_mbc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_mbc)
        print('\n')
        print('Unique Values:')
        print(mbc_predict_test_global.unique())

    def main(self):
        self.MiniBatchSGDClassifier()

In [ ]:
ml_MBSGDClassifier = machine_learing(df, generated_data_global)
ml_MBSGDClassifier.main()
metrics_MBSGDClassifier = Metrics()
print("Metrics for not normalized data is ready. Run 'metrics_MBSGDClassifier.main()'!")
#metrics_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_maxAbsScaler_MBSGDClassifier = machine_learing(maxAbsScaler_merged_global, generated_data_global)
ml_maxAbsScaler_MBSGDClassifier.main()
metrics_maxAbsScaler_MBSGDClassifier = Metrics()
print("Metrics for maxAbsScaler normalized data is ready. Run 'metrics_maxAbsScaler_MBSGDClassifier.main()'!")
#metrics_maxAbsScaler_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_MinMaxScaler_MBSGDClassifier = machine_learing(MinMaxScaler_merged_global, generated_data_global)
ml_MinMaxScaler_MBSGDClassifier.main()
metrics_MinMaxScaler_MBSGDClassifier = Metrics()
print("Metrics for MinMaxScaler normalized data is ready. Run 'metrics_MinMaxScaler_MBSGDClassifier.main()'!")
#metrics_MinMaxScaler+MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_normalizer_MBSGDClassifier = machine_learing(normalizer_merged_global, generated_data_global)
ml_normalizer_MBSGDClassifier.main()
metrics_normalizer_MBSGDClassifier = Metrics()
print("Metrics for normalizer normalized data is ready. Run 'metrics_normalizer_MBSGDClassifier.main()'!")
#metrics_normalizer_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_robust_scaler_MBSGDClassifier = machine_learing(robust_scaler_merged_global, generated_data_global)
ml_robust_scaler_MBSGDClassifier.main()
metrics_robust_scaler_MBSGDClassifier = Metrics()
print("Metrics for robust_scaler normalized data is ready. Run 'metrics_robust_scaler_MBSGDClassifier.main()'!")
#metrics_robust_scaler_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_standard_scaler_MBSGDClassifier = machine_learing(standard_scaler_merged_global, generated_data_global)
ml_standard_scaler_MBSGDClassifier.main()
metrics_standard_scaler_MBSGDClassifier = Metrics()
print("Metrics for standard_scaler normalized data is ready. Run 'metrics_standard_scaler_MBSGDClassifier.main()'!")
#metrics_standard_scaler_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_binarizer_MBSGDClassifier = machine_learing(binarizer_merged_global, generated_data_global)
ml_binarizer_MBSGDClassifier.main()
metrics_binarizer_MBSGDClassifier = Metrics()
print("Metrics for binarizer normalized data is ready. Run 'metrics_binarizer_MBSGDClassifier.main()'!")
#metrics_binarizer_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_function_transformer_MBSGDClassifier = machine_learing(function_transformer_merged_global, generated_data_global)
ml_function_transformer_MBSGDClassifier.main()
metrics_function_transformer_MBSGDClassifier = Metrics()
print("Metrics for function_transformer normalized data is ready. Run 'metrics_function_transformer_MBSGDClassifier.main()'!")
#metrics_function_transformer_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
ml_KBinsDiscretizer_MBSGDClassifier = machine_learing(KBinsDiscretizer_merged_global, generated_data_global)
ml_KBinsDiscretizer_MBSGDClassifier.main()
metrics_KBinsDiscretizer_MBSGDClassifier = Metrics()
print("Metrics for KBinsDiscretizer normalized data is ready. Run 'metrics_KBinsDiscretizer_MBSGDClassifier.main()'!")
#metrics_KBinsDiscretizer_MBSGDClassifier.main()

In [ ]:
torch.cuda.empty_cache()
gc.collect()